# 02 · State

**Goal:** by the end you can describe the calculator's state with a `TypedDict`, read values from it, and choose
between `TypedDict`, `dataclass` and Pydantic `BaseModel` for a state schema.

## The idea in plain English

In lesson 01 the state was a dict we filled in as we went. Nothing said which keys it *should* have. In LangGraph
you write that down first, in a **state schema**: a short class that lists every key and its type. LangGraph reads
the schema to know which keys exist, and every node receives the whole state and can read any key in it.

For the next few lessons the calculator's state has three keys: `a` and `b` (the two numbers) and `result` (the
answer). This lesson is only about describing and reading state. We start building graphs with it in lesson 04.

**Analogy:** a paper form with labelled boxes. The schema is the blank form: it says which boxes exist and what goes
in each (a number, some text). A filled-in form is one state.

## Picture

One state, three keys. Any node can read any of them. (The `add` node arrives in lesson 03.)

```mermaid
flowchart LR
    subgraph CalcState
        a["a: float = 3.0"]
        b["b: float = 4.0"]
        result["result: float = 7.0"]
    end
    CalcState -. "every node can read every key" .-> node["a node (lesson 03)"]
```

## Step 1: Describe the state with `TypedDict`

A **`TypedDict`** is a dict with named keys and declared types. It's the most common way to write a LangGraph state
schema, and the one this tutorial uses. Each line inside the class is `key_name: type`.

The class keeps its list of keys and types in `__annotations__`, so we can print it.

In [1]:
from typing import TypedDict


class CalcState(TypedDict):
    a: float
    b: float
    result: float


for key, kind in CalcState.__annotations__.items():
    print(f"{key:7} -> {kind.__name__}")

a       -> float
b       -> float
result  -> float


## Step 2: A state is still just a dict

You make a state with ordinary dict syntax. The `: CalcState` after the variable name is a **type hint**: a note for
you and for code editors saying what shape the dict should have. At run time the value is a plain `dict`.

In [2]:
state: CalcState = {"a": 3.0, "b": 4.0, "result": 0.0}

print(state)
print("type:", type(state).__name__)

{'a': 3.0, 'b': 4.0, 'result': 0.0}
type: dict


## Step 3: Read values from the state

You read a key with square brackets, `state["a"]`, exactly as with any dict. This function receives the whole state
and uses two of its keys. Every node in LangGraph will look like this: the state comes in, the node reads what it
needs.

In [3]:
def describe(state: CalcState) -> str:
    return f"a is {state['a']}, b is {state['b']}, a + b would be {state['a'] + state['b']}"


print(describe(state))

a is 3.0, b is 4.0, a + b would be 7.0


## Step 4: Reading a key that isn't filled in yet

At the start of a run, `result` hasn't been calculated, so the state usually has only `a` and `b`. Square brackets
fail on a missing key. `state.get("result")` returns `None` instead, and `state.get("result", 0.0)` returns a
default you choose. Use `.get` when a key might not be there yet.

In [4]:
start: CalcState = {"a": 3.0, "b": 4.0}  # no "result" yet

print("get without default:", start.get("result"))
print("get with default:   ", start.get("result", 0.0))
print("'result' in start:  ", "result" in start)

get without default: None
get with default:    0.0
'result' in start:   False


## Step 5: `TypedDict` does not check types

The types in a `TypedDict` are only notes. Python does **not** check them when the program runs. Below, `a` is the
text `"3"` instead of a number. Making the state works fine. The mistake only shows up later, when some code tries
to add text to a number. Keep this in mind: it's the main reason to consider Pydantic (Step 7).

In [5]:
bad: CalcState = {"a": "3", "b": 4.0, "result": 0.0}  # "3" is text, not a number
print("Accepted without complaint:", bad)
print("type of a:", type(bad["a"]).__name__)

try:
    print(bad["a"] + bad["b"])
except TypeError as err:
    print("Later, adding fails: TypeError:", err)

Accepted without complaint: {'a': '3', 'b': 4.0, 'result': 0.0}
type of a: str
Later, adding fails: TypeError: can only concatenate str (not "float") to str


## Step 6: Alternative 1, a `dataclass`

A **`dataclass`** is a small class that stores named fields. Two differences from `TypedDict`:

- You read fields with a dot, `s.a`, not `s["a"]`.
- Fields can have **default values**. Here `result` starts at `0.0` if you don't give one.

LangGraph accepts a dataclass as a state schema. The LangGraph docs suggest it when you want defaults.

In [6]:
from dataclasses import dataclass


@dataclass
class CalcStateDC:
    a: float
    b: float
    result: float = 0.0  # default value


s = CalcStateDC(a=3.0, b=4.0)
print(s)
print("s.a + s.b =", s.a + s.b)

CalcStateDC(a=3.0, b=4.0, result=0.0)
s.a + s.b = 7.0


## Step 7: Alternative 2, a Pydantic `BaseModel`

**Pydantic** is a library that **validates** data, which means it checks every value against its type when the
object is created. (LangChain installs it, so it's already in your venv.) A Pydantic state class inherits from
`BaseModel`.

Validation does two things. It converts values that can be safely converted, so the text `"3"` becomes the number
`3.0`. And it rejects values that can't be converted, raising a `ValidationError` that names the bad field.

In [7]:
from pydantic import BaseModel, ValidationError


class CalcStateModel(BaseModel):
    a: float
    b: float
    result: float = 0.0


m = CalcStateModel(a="3", b=4)  # "3" and 4 are converted to floats
print(m)
print("type of a:", type(m.a).__name__)

a=3.0 b=4.0 result=0.0
type of a: float


Now some input that can't be a number. Pydantic stops it right away, at the door, instead of letting it crash some
later step the way the `TypedDict` did in Step 5.

In [8]:
try:
    CalcStateModel(a="three", b=4)
except ValidationError as err:
    print(err)

1 validation error for CalcStateModel
a
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='three', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


## Which one should I use?

| Schema type | Read a value | Defaults | Checks types at run time | Pick it when |
|---|---|---|---|---|
| `TypedDict` | `state["a"]` | No | No | Almost always. It's the LangGraph default and the fastest. |
| `dataclass` | `state.a` | Yes | No | You want default values for some keys. |
| Pydantic `BaseModel` | `state.a` | Yes | **Yes** | Input comes from users or other systems and must be validated. |

The LangGraph docs note that Pydantic is slower than the other two, because it checks every value. This tutorial
uses `TypedDict` from here on. Whichever you choose, nodes still *return* a plain dict of updates (lesson 03).

## What just happened

- **Step 1** declared `CalcState` and printed its schema: `a -> float`, `b -> float`, `result -> float`.
- **Step 2** made a state and showed its type is just `dict`.
- **Step 3** read two keys from the state inside a function, the way every node will.
- **Step 4** showed `.get("result")` returns `None` for a key that isn't filled in yet, instead of failing.
- **Step 5** showed `TypedDict` accepted `a="3"` without complaint. The error only came later: `TypeError: can only
  concatenate str (not "float") to str`.
- **Step 6** used a `dataclass`, read fields with a dot, and got `result=0.0` from the default.
- **Step 7** used Pydantic: `a="3"` became `3.0`, and `a="three"` was rejected immediately with a `ValidationError`.

## Common mistakes

1. **Using a dot on a `TypedDict` state.** A `TypedDict` state is a dict, so `state.a` fails:
   `AttributeError: 'dict' object has no attribute 'a'`.
   Fix: use `state["a"]`. (The dot works only for `dataclass` and Pydantic states.)
2. **Reading a key that hasn't been set.** `state["result"]` before anything wrote `result` fails:
   `KeyError: 'result'`.
   Fix: use `state.get("result")`, or make sure an earlier node writes that key.
3. **Expecting `TypedDict` to catch bad values.** It never does. The error appears far from its cause, for example
   `TypeError: can only concatenate str (not "float") to str`.
   Fix: use a Pydantic `BaseModel` when input needs checking.

## Try it

1. The calculator will soon need to know *which* operation to do. Write `CalcState2`, a `TypedDict` with `a`, `b`,
   `op: str` and `result`. Make a state for `6 * 7` and print a sentence like `6.0 * 7.0` using `state["op"]`.
2. Using `CalcStateModel` from Step 7, predict what happens with `b="0.5"` and with `b=None`, then run both to
   check.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **State schema**: a class listing the keys of the state and their types.
- **`TypedDict`**: a dict with named, typed keys. The default way to write a LangGraph state schema.
- **Type hint**: a note like `: float` or `: CalcState` saying what type a value should be. Python doesn't check it.
- **`dataclass`**: a small class with named fields and optional default values.
- **Pydantic `BaseModel`**: a class that validates its values when it's created.
- **Validation**: checking values against their types, converting them where safe and rejecting them otherwise.